# Zero-Shot Classification with AutoModel



In [ ]:
pip install --upgrade transformers sympy

In [11]:
pip install pip install --upgrade transformers sympy

ERROR: Could not find a version that satisfies the requirement install (from versions: none)
ERROR: No matching distribution found for install


In [20]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "facebook/bart-large-mnli"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
print('ID --> Label: ' + str(model.config.id2label))

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

ID --> Label: {0: 'contradiction', 1: 'neutral', 2: 'entailment'}


In [29]:
def yeehaw(sequence, labels, multi_label=False, hypothesis_template="This text is about {}."):
  # Zero-shot classification works by comparing a "premise" to a "hypothesis"...
  # Create a list of hypotheses and a list of premises
  hypotheses = [hypothesis_template.format(label) for label in labels]
  premises = [sequence] * len(labels)

  # 2. Tokenize as premise/hypothesis pairs
  inputs = tokenizer(premises, hypotheses, return_tensors="pt", padding=True)
  # print("Inputs (token IDs): \n" + str(inputs))

  # Convert token IDs back to raw tokens, for the sake of learning (=
  raw_tokens = [tokenizer.convert_ids_to_tokens(ids) for ids in inputs.input_ids]
  # print("Inputs (raw tokens): \n" + str(raw_tokens))

  # 3. Run model
  with torch.no_grad():
    outputs = model(**inputs)
    # print("Outputs: \n" + str(outputs))

  # 4. Extract "Entailment" scores
  # For BART/DeBERTa MNLI models, index 2 is usually 'entailment
  # We can verify this with the following (executed in cell above):
  #    print('ID --> Label: ' + str(model.config.id2label))
  entailment_logits = outputs.logits[:, 2]
  if (not multi_label):
    # Multi-class (Single choice): Use softmax over all
    # entailment logits so the total probability sums to 1.
    scores = torch.softmax(entailment_logits, dim=0)
  else:
    # Multi-label (Multiple choices): If a sequence can belong to "finance" and "technology"
    # simultaneously, use sigmoid on each individual entailment logit instead of softmax.
    scores = torch.sigmoid(entailment_logits)

  # 5. Map scores back to labels
  result = dict(zip(labels, scores.tolist()))
  # print('Result: \n' + str(result))
  result_sorted = sorted(result.items(), key=lambda x: x[1], reverse=True)
  print('"' + sequence + '"' + ' is a ' + result_sorted[0][0] + ' thing to say!')


yeehaw('Your bosom would look absolutely scrumptious adorned with a white pearly necklace', ['kiddy', 'fancy', 'curmudgeony'])

"Your bosom would look absolutely scrumptious adorned with a white pearly necklace" is a fancy thing to say!


# What's going on here? What is an 'entailment'?

In the world of Natural Language Processing, Entailment comes from a task called Natural Language Inference (NLI), also known as Recognizing Textual Entailment (RTE).
In NLI, a model is given two sentences—a Premise and a Hypothesis—and must decide the logical relationship between them. There are typically three possible labels:

   1. Entailment (Index 2): If the Premise is true, the Hypothesis must also be true. (Logical "Yes")
   2. Neutral (Index 1): The Premise doesn't give enough information to confirm or deny the Hypothesis. (Logical "Maybe")
   3. Contradiction (Index 0): If the Premise is true, the Hypothesis must be false. (Logical "No")

## Why it’s used for Zero-Shot
Zero-shot classification "tricks" an NLI model into doing classification.

* Premise: Your input text (e.g., "The stock market crashed.")
* Hypothesis: A template containing your label (e.g., "This text is about finance.")

The model then asks itself: "If the premise is true, does it logically follow that the hypothesis is true?"

* If the model predicts Entailment, it means the label "finance" is a highly accurate description of the text.
* The higher the score for Index 2, the more confident the model is that the label fits.

## Why "Index 2"?
Most popular NLI datasets (like MNLI) are structured so that the output layer has three nodes. For models like facebook/bart-large-mnli, the mapping is strictly:

* 0: Contradiction
* 1: Neutral
* 2: Entailment

When you extract logits[:, 2], you are specifically grabbing the "Yes, this matches" score while ignoring the "Maybe" and "No" scores.
Do you want to see how to verify the label mapping for a specific model using model.config.label2id?

